# Homework 2 — Colab starter

Complete the assigned preparation before beginning.

## How to work in this notebook

This `.ipynb` is your **only working document**. The assignment PDF is a read-only copy of the same prompt. Do not edit or combine a `.qmd` file.

- Write reasoning in the designated text cells.
- Run or modify the starter code rather than pasting an unexplained replacement.
- Keep requested output, plots, excerpts, and raw AI evidence visible.
- Handwriting is welcome but never required. Typed Markdown/LaTeX is fully equivalent.
- If you insert a clear scan/photo, add one typed description or statistical conclusion for accessibility.

Before submission, restart and run all. Upload a PDF export and the completed `.ipynb` to the same Gradescope assignment. If image insertion fails, add one clearly labeled optional handwriting PDF; do not merge files.

In [ ]:
# Standard Colab setup - run once
from pathlib import Path
import hashlib
import json
import random
import sys
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2027
random.seed(SEED)
np.random.seed(SEED)

COURSE_REPO_RAW_URL = 'https://raw.githubusercontent.com/skgallagher/stat-methods-ai-public/main'
COURSE_DATA_BASE_URL = COURSE_REPO_RAW_URL + '/data/course'
COURSE_DATA_GROUPS = ['dynasent']
DATA_ROOT = Path('/content/stat_ai_data')

# Local repository runs use the frozen release when present and otherwise the
# synthetic smoke fixture. A fresh Colab downloads verified individual files
# from GitHub - no ZIP upload or Drive mount is required.
LOCAL_RELEASE = Path.cwd() / 'data' / 'course'
LOCAL_SMOKE = Path.cwd() / 'data' / 'smoke'
online_release = False
if (LOCAL_RELEASE / 'manifest.json').exists():
    DATA_ROOT = LOCAL_RELEASE
    data_source = 'local frozen release'
elif LOCAL_SMOKE.exists():
    DATA_ROOT = LOCAL_SMOKE
    data_source = 'local synthetic smoke fixture (development only)'
elif (DATA_ROOT / 'manifest.json').exists():
    cached_manifest = json.loads((DATA_ROOT / 'manifest.json').read_text())
    if 'smoke fixture' in cached_manifest.get('bundle_type', ''):
        data_source = 'existing local synthetic smoke fixture (development only)'
    else:
        cached_files = cached_manifest.get('files', [])
        requested_files = [
            item for item in cached_files
            if Path(item['path']).parts[0] in COURSE_DATA_GROUPS
        ]
        def cached_sha256(path):
            digest = hashlib.sha256()
            with path.open('rb') as stream:
                for chunk in iter(lambda: stream.read(1024 * 1024), b''):
                    digest.update(chunk)
            return digest.hexdigest()
        cache_complete = (
            cached_manifest.get('release_status') == 'student_release'
            and requested_files
            and all(
                (DATA_ROOT / item['path']).exists()
                and cached_sha256(DATA_ROOT / item['path']) == item['sha256']
                for item in requested_files
            )
        )
        if cache_complete:
            data_source = 'existing verified runtime cache'
        else:
            online_release = True
else:
    online_release = True

if online_release:
    helper_target = Path('/content/course_helpers/__init__.py')
    helper_target.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        COURSE_REPO_RAW_URL + '/course_helpers/__init__.py', helper_target
    )
    if '/content' not in sys.path:
        sys.path.insert(0, '/content')
    from course_helpers import ensure_course_data
    DATA_ROOT = ensure_course_data(
        COURSE_DATA_BASE_URL,
        DATA_ROOT,
        groups=COURSE_DATA_GROUPS,
    )
    data_source = 'public GitHub student release'

print('Setup complete. Data root:', DATA_ROOT)
print('Data source:', data_source)
print('Requested groups:', COURSE_DATA_GROUPS)


| | |
|---|---|
| **Out / due** | Tue Jan 26 / Tue Feb 2, 11:59pm |
| **Points** | 100: Problem 1 (50), Problem 2 (30), Problem 3 (20) |
| **Expected time** | About 5 hours after completing Lab 2; contact course staff if setup/debugging alone exceeds 30 minutes |
| **Reading** | Plank, paper pp. 1–5; DynaSent Figure 1 and §§3.3–3.4, 4.2–4.5; Shalizi, “Some basic math about averaging” and “Exchangeable variables and random limits for averages”; the short Statistical Proof Book proof of the law of total covariance; Bishop & Bishop Chapter 5 and Exercise 5.1 as technical support |
| **AI policy** | No AI for Problem 1. AI is optional in Problem 2(a–d) and required for the 5-point attempt in Problem 2(e) and for Problem 3. Submit the AI-use record at the end. |
| **Working file** | Complete all work in `hw02_starter.ipynb`; the PDF is a read-only prompt copy. |
| **Submit** | Notebook PDF + completed `.ipynb` in one Gradescope submission. |

## Connection to Lab 2

Lecture introduced labels as measurements produced by a protocol. Lab 2 used a
DynaSent teaching subset to inspect disagreement, compare construction rounds,
and construct hard and soft outcomes from five rater labels. This homework asks
you to explain the theory behind that construction, examine what aggregation and
item selection hide, and evaluate forecasts against targets made from new,
disjoint DynaSent measurements.

## The throughline: one sentence, five different objects

The central question is **what, exactly, is the target?** Keep these objects
separate throughout the homework:

| Object | Role in this homework |
|---|---|
| $L_i$ | one rater's categorical measurement of a sentence |
| $q$ | the response distribution induced by the working population and labeling model |
| $\widehat q$ | the empirical soft target constructed from the observed rater measurements |
| $Y_{\mathrm{maj}}$ | the hard target constructed by majority vote |
| $p$ | a forecaster's probability report, made before observing the measurements |

The working model does not declare one observed label to be revealed truth.
Instead, it lets us ask what repeated labels estimate, what majority vote
discards, and how a constructed target shapes model evaluation.

You may reuse the lab's rater-expansion, vote-summary, Brier-score, and
denominator-first table code. The homework uses different items and asks for a
new statistical argument. As self-checks, every item must have exactly five
votes, the three rater proportions must sum to one, and every unnormalized
three-class Brier score must lie between 0 and 2.

In [ ]:
# HW2: rater summaries, Brier targets, and aligned forecasters
from course_helpers import expand_dynasent_raters, summarize_votes
LABELS = ['negative', 'neutral', 'positive']
def multiclass_brier(prob, outcome):
    prob, outcome = np.asarray(prob), np.asarray(outcome)
    return np.sum((prob - outcome) ** 2)
dynasent_root = DATA_ROOT / 'dynasent'
homework_items = pd.read_csv(dynasent_root / 'homework_items.csv')
forecaster_items = pd.read_csv(dynasent_root / 'forecaster_items.csv')
ratings = expand_dynasent_raters(homework_items)
rater_summary = summarize_votes(ratings, homework_items)
assert ratings.groupby('item_id').size().eq(5).all()
assert np.allclose(rater_summary[[f'p_{label}' for label in LABELS]].sum(axis=1), 1)
# Problem 1 is mathematical work; complete it in the response cells without AI.
# Problem 2 begins with rater_summary. Make denominator-first vote-pattern
# and round tables using the lab workflow, then add the requested figure.
# Problem 3 initially reveals only the IDs and sentences.
# Do not inspect the vote columns until the reveal cell in part (c).
forecaster_items[['item_id', 'sentence']]

# Problem 1 — A label is a random outcome (50 points)

Complete this problem without AI assistance. Show the mathematical steps that
support each conclusion; isolated numerical answers receive little credit.
Parts (a)(i)–(iii) begin with Bishop & Bishop Exercise 5.1 and extend it to the
Brier score used in lecture and lab.

**Notation and working model.** Fix a sentence $x$. Let $L$ be a random label
generated according to the working model

$$
L\mid x\sim \operatorname{Categorical}
  (q_{\mathrm{neg}},q_{\mathrm{neu}},q_{\mathrm{pos}}).
$$

Thus $q_k=P(L=k\mid x)$ is the conditional probability of label $k$ under the
working model, and the three probabilities are nonnegative and sum to one.
For each class $k$, define the indicator

$$T_k=\mathbf 1\{L=k\},$$

which equals one when $L=k$ and zero otherwise. The vector
$T=(T_{\mathrm{neg}},T_{\mathrm{neu}},T_{\mathrm{pos}})$ is the one-hot
encoding of $L$: exactly one component equals one. For example, a positive
label gives $T=(0,0,1)$. Suppose we must choose a probability vector
$p=(p_{\mathrm{neg}},p_{\mathrm{neu}},p_{\mathrm{pos}})$ to report before
observing $L$. In this derivation, $p$ is a fixed candidate report; the
expectation is over the random label $L\mid x$. The three-class Brier score is

$$B(p,T)=\sum_{k=1}^{3}(p_k-T_k)^2.$$

Each numbered item is a separate grading checkpoint. Show your setup and
intended next step; correct partial work can earn substantial credit.

a. **One categorical label and its expected Brier score (20 points).**

   **(i) Categorical label and indicators (4 points).** Write the three possible
   values of $L$ and the corresponding one-hot vectors $T$. Explain why the
   three indicators cannot be treated as independent Bernoulli random
   variables even though each indicator is zero or one.

   **(ii) Expected value of an indicator (4 points).** Use the definition of
   expected value for a zero-one variable to complete

$$
\begin{aligned}
E[T_k\mid x]
  &=1\cdot P(T_k=1\mid x)+0\cdot P(T_k=0\mid x)\\
  &=\underline{\hspace{4cm}}.
\end{aligned}
$$

   Then use $T_k=\mathbf 1\{L=k\}$ to show that
   $E[T_k\mid x]=P(L=k\mid x)=q_k$.

   **(iii) One-class squared loss (6 points).** Hold $p_k$ fixed and show that

$$
E[(p_k-T_k)^2\mid x]
  =(p_k-q_k)^2+q_k(1-q_k).
$$

   **(iv) Sum, interpret, and minimize (6 points).** Show that summing part (iii)
   over the three classes gives

$$
E[B(p,T)\mid x]
  =\sum_{k=1}^{3}(p_k-q_k)^2+\sum_{k=1}^{3}q_k(1-q_k).
$$

   In 1–2 sentences, explain what each of the two sums represents and identify
   which one depends on the candidate report $p$. Find the value of $p$ that
   minimizes the expected Brier score under this model, and explain what your
   answer means in context.

### Your response — Problem 1(a)

Type your response here **or** insert a clear image of handwritten work here.

**Prediction before assistance/output, when requested:**  
TODO or not applicable

**Analysis, evidence, or reasoning:**  
TODO

**Typed statistical conclusion (required even with handwriting):**  
TODO

b. **Repeated labels estimate class probabilities (15 points).** For the same
sentence $x$, suppose we observe $m$ conditionally independent labels
$L_1,\ldots,L_m$ from the working categorical model with common probability
vector $q$. Define

$$
T_{ik}=\mathbf 1\{L_i=k\},\qquad
N_k=\sum_{i=1}^{m}T_{ik},\qquad
\widehat q_k=\frac{N_k}{m}.
$$

   **(i) Structural identities (3 points).** Show that
   $\sum_{k=1}^{3}N_k=m$ and $\sum_{k=1}^{3}\widehat q_k=1$.

   **(ii) Mean and variance (6 points).** Use indicator expectations and the
   conditional independence assumption to show that

$$
E[\widehat q_k\mid x]=q_k,
\qquad
\operatorname{Var}(\widehat q_k\mid x)=\frac{q_k(1-q_k)}{m}.
$$

   **(iii) Dependence across classes (4 points).** For $j\ne k$, use the fact
   that $T_{ij}T_{ik}=0$ to show that

$$
\operatorname{Cov}(\widehat q_j,\widehat q_k\mid x)
=-\frac{q_jq_k}{m}.
$$

   In one sentence, explain why the covariance is negative.

   **(iv) More labels (2 points).** In one sentence, explain what changes—and
   what does not change—as $m$ increases.

### Your response — Problem 1(b)

Type your response here **or** insert a clear image of handwritten work here.

**Prediction before assistance/output, when requested:**  
TODO or not applicable

**Analysis, evidence, or reasoning:**  
TODO

**Typed statistical conclusion (required even with handwriting):**  
TODO

c. **Expected Brier score for an empirical soft target (15 points).** Continue
to use the model and notation from part (b), and treat $p$ as a fixed candidate
report.

   **(i) One class (6 points).** Use the mean and variance from part (b) to show
   that

$$
E[(p_k-\widehat q_k)^2\mid x]
=(p_k-q_k)^2+\frac{q_k(1-q_k)}{m}.
$$

   **(ii) Sum and compare (5 points).** Sum over the three classes. Then compare
   the result with part (a), where the Brier score was computed against one
   random categorical label. In 1–2 sentences, explain what the factor $1/m$
   tells you.

   **(iii) Minimize and take a limit (4 points).** Find the candidate report $p$
   that minimizes the expected Brier score for any positive integer $m$. Then
   find the limit of the expected score as $m\to\infty$ and explain it in one
   sentence.

### Your response — Problem 1(c)

Type your response here **or** insert a clear image of handwritten work here.

**Prediction before assistance/output, when requested:**  
TODO or not applicable

**Analysis, evidence, or reasoning:**  
TODO

**Typed statistical conclusion (required even with handwriting):**  
TODO

# Problem 2 — A benchmark label is constructed, not found (30 points)

Use the 72 items in `homework_items.csv`. Work at the **sentence level**: five
rater rows are repeated measurements of one item, not five independent
evaluation cases. The majority label is an aggregation rule applied to those
measurements; it is not an additional observation.

a. **Where disagreement occurs (6 points).** Describe the distribution of vote
patterns overall and separately for Rounds 1 and 2. Include one table with the
item count, round total, and within-round proportion for every
round-by-pattern combination, plus one figure showing the same comparison. In
1–2 sentences, describe the main pattern. Keep every denominator visible.

### Your response — Problem 2(a)

Type your response here **or** insert a clear image of handwritten work here.

**Prediction before assistance/output, when requested:**  
TODO or not applicable

**Analysis, evidence, or reasoning:**  
TODO

**Typed statistical conclusion (required even with handwriting):**  
TODO

b. **Performance by vote pattern (7 points).** For `5-0`, `4-1`, and `3-2`
items, report the number of items, number correctly classified under the
majority label, and majority-label accuracy. In 2–3 sentences, describe the
comparison and explain why rater disagreement is not automatically “bad data.”
Do not add confidence intervals; interval construction begins in Week 4.

### Your response — Problem 2(b)

Type your response here **or** insert a clear image of handwritten work here.

**Prediction before assistance/output, when requested:**  
TODO or not applicable

**Analysis, evidence, or reasoning:**  
TODO

**Typed statistical conclusion (required even with handwriting):**  
TODO

c. **Rounds are construction processes, not treatments (7 points).** Compare
Round 1 and Round 2 using majority-label accuracy. Report the item count, number
correct, accuracy, and Round 2 minus Round 1 difference. Then write 3–4
sentences that:

1. state what the assigned data show;
2. use one specific detail from the assigned DynaSent documentation to explain
   how the rounds were constructed;
3. give two plausible explanations for the observed difference; and
4. explain why the comparison does not identify a causal effect of collection
   round.

### Your response — Problem 2(c)

Type your response here **or** insert a clear image of handwritten work here.

**Prediction before assistance/output, when requested:**  
TODO or not applicable

**Analysis, evidence, or reasoning:**  
TODO

**Typed statistical conclusion (required even with handwriting):**  
TODO

d. **One planned sensitivity analysis (5 points).** Before computing the
alternative result, choose and record **one** of these changes:

- keep all items and replace majority-label accuracy with mean Brier score
  against the rater proportions; or
- keep majority-label accuracy and exclude all `3-2` items.

   Complete the starter comparison table with each analysis's outcome, retained
   population, round estimates, and Round 2 minus Round 1 difference. In 2–3
   sentences, state what changed and whether the substantive comparison
   survived.

### Your response — Problem 2(d)

**Chosen alternative before computing:**  
TODO — choose exactly one of the two listed changes

| Analysis | Outcome | Retained population | Round 1 estimate | Round 2 estimate | Round 2 minus Round 1 |
|---|---|---|---:|---:|---:|
| Primary | Majority-label accuracy | All 72 assigned items | TODO | TODO | TODO |
| Planned alternative | TODO | TODO | TODO | TODO | TODO |

**Interpretation (2-3 sentences):**  
TODO — state what changed and whether the substantive comparison survived

e. **Red-team the leaderboard with AI (5 points).** Attempting this part is
required to earn these 5 points; successfully increasing accuracy is not
required. Tell an AI assistant that its job is to make the fixed model's reported
majority-label accuracy look better **without changing the model**. Ask it for
one transparent, superficially defensible rule that filters items using only
sentence text or rater-disagreement information. The rule may not use model
probabilities, model predictions, correctness, or the result of part (d), and
you may not revise it after seeing the answer.

   Preserve the initial prompt, state the rule and predicted direction before
   computing, then apply it once. Report the item count, number correct, and
   accuracy before and after filtering. In 1–2 sentences, explain what changed,
   why the model itself did not improve, and why this would be a questionable
   way to build a benchmark. Connect your explanation to the throughline: the
   selection rule changed which measured items became the evaluation target.

### Your response — Problem 2(e)

Type your response here **or** insert a clear image of handwritten work here.

**Prediction before assistance/output, when requested:**  
TODO or not applicable

**Analysis, evidence, or reasoning:**  
TODO

**Typed statistical conclusion (required even with handwriting):**  
TODO

# Problem 3 — Three forecasters, one measured target (20 points)

Use the ten purposefully selected DynaSent test sentences in
`forecaster_items.csv`. These cases were selected to make disagreement visible;
they are a demonstration, not a probability sample. The five DynaSent labels
are the measurements. You and the two frozen classifiers are **forecasters** of
a target constructed from those measurements:

- [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest),
  revision `3216a57f`;
- [`lxyuan/distilbert-base-multilingual-cased-sentiments-student`](https://huggingface.co/lxyuan/distilbert-base-multilingual-cased-sentiments-student),
  revision `cf991100`.

The starter notebook downloads the public checkpoints and runs them locally;
no paid API is required. Do not inspect the five-rater vote columns before
completing part (a).

a. **Be the first forecaster (4 points).** Before running either model, report
your own probability vector over `negative`, `neutral`, and `positive` for every
sentence. Each row must be nonnegative and sum to one. Add your highest-
probability label and identify the two sentences for which you were least
certain, with one short text-based reason for each.

In [ ]:
# Fill one probability per sentence in the displayed row order before running either model.
student_forecasts = forecaster_items[['item_id', 'sentence']].copy()
student_forecasts['student_p_negative'] = [np.nan] * len(student_forecasts)
student_forecasts['student_p_neutral'] = [np.nan] * len(student_forecasts)
student_forecasts['student_p_positive'] = [np.nan] * len(student_forecasts)
# Replace all 30 np.nan entries with your own probabilities, then run this cell.
# Each row must contain three nonnegative numbers that sum to one.
student_forecasts

### Your response — Problem 3(a)

Type your response here **or** insert a clear image of handwritten work here.

**Prediction before assistance/output, when requested:**  
TODO or not applicable

**Analysis, evidence, or reasoning:**  
TODO

**Typed statistical conclusion (required even with handwriting):**  
TODO

b. **Run two frozen models (5 points).** Run the supplied checkpoint cell and
preserve the raw probability output. Complete the supplied ten-row table with
both models' three probabilities, highest-probability labels, and a
`models_disagree` indicator. Show that both probability vectors sum to one for
every row and report the number of disagreements out of ten. Do not interpret
that fraction as a population estimate.

In [ ]:
%pip -q install 'transformers==5.13.0'
from transformers import pipeline
STUDENT_PROB_COLS = [f'student_p_{label}' for label in LABELS]
assert student_forecasts[STUDENT_PROB_COLS].notna().all().all(), (
    'Complete your own forecasts before running the models.'
)
assert np.allclose(student_forecasts[STUDENT_PROB_COLS].sum(axis=1), 1)
student_forecasts['student_label'] = (
    student_forecasts[STUDENT_PROB_COLS].idxmax(axis=1).str.removeprefix('student_p_')
)
MODEL_SPECS = {
    'cardiff': (
        'cardiffnlp/twitter-roberta-base-sentiment-latest',
        '3216a57f2a0d9c45a2e6c20157c20c49fb4bf9c7'),
    'distilbert': (
        'lxyuan/distilbert-base-multilingual-cased-sentiments-student',
        'cf991100d706c13c0a080c097134c05b7f436c45'),
}
def run_frozen_forecaster(short_name, sentences):
    model_id, revision = MODEL_SPECS[short_name]
    classifier = pipeline(
        'text-classification', model=model_id, revision=revision,
        top_k=None, device=-1)
    raw = classifier(sentences, truncation=True, batch_size=10)
    rows = []
    for result in raw:
        scores = {entry['label'].lower(): entry['score'] for entry in result}
        rows.append({
            **{f'{short_name}_p_{label}': scores[label] for label in LABELS},
            f'{short_name}_label': max(LABELS, key=scores.get),
        })
    return pd.DataFrame(rows)
model_outputs = forecaster_items[['item_id', 'sentence']].copy()
for short_name in MODEL_SPECS:
    model_outputs = pd.concat([
        model_outputs,
        run_frozen_forecaster(short_name, model_outputs['sentence'].tolist()),
    ], axis=1)
for short_name in MODEL_SPECS:
    cols = [f'{short_name}_p_{label}' for label in LABELS]
    assert np.allclose(model_outputs[cols].sum(axis=1), 1)
model_outputs['models_disagree'] = (
    model_outputs['cardiff_label'] != model_outputs['distilbert_label']
)
model_outputs

### Your response — Problem 3(b)

Type your response here **or** insert a clear image of handwritten work here.

**Prediction before assistance/output, when requested:**  
TODO or not applicable

**Analysis, evidence, or reasoning:**  
TODO

**Typed statistical conclusion (required even with handwriting):**  
TODO

c. **Compare two constructed targets (5 points).** Reveal the vote counts and
construct both $\widehat q$, the three rater proportions, and
$Y_{\mathrm{maj}}$, its one-hot majority-vote version. For you and each model,
report the mean Brier score against **each** target and the number of
highest-probability labels matching the majority; keep the denominator of ten
visible. Briefly explain what information is lost when $\widehat q$ is replaced
by $Y_{\mathrm{maj}}$ and why neither target becomes literal ground truth.

In [ ]:
# Reveal the five measurements only after locking all three forecasts.
target = forecaster_items.copy()
for label in LABELS:
    target[f'q_{label}'] = target[f'{label}_votes'] / 5
TARGET_COLS = [f'q_{label}' for label in LABELS]
assert target[[f'{label}_votes' for label in LABELS]].sum(axis=1).eq(5).all()
assert np.allclose(target[TARGET_COLS].sum(axis=1), 1)
target['majority_label'] = target[TARGET_COLS].idxmax(axis=1).str.removeprefix('q_')
comparison = (
    student_forecasts.merge(model_outputs, on=['item_id', 'sentence'], validate='one_to_one')
    .merge(target, on=['item_id', 'sentence'], validate='one_to_one')
)
soft_target = comparison[TARGET_COLS].to_numpy()
majority_target = np.column_stack([
    comparison['majority_label'].eq(label).astype(float) for label in LABELS
])
summary_rows = []
for forecaster in ['student', 'cardiff', 'distilbert']:
    prob_cols = [f'{forecaster}_p_{label}' for label in LABELS]
    probabilities = comparison[prob_cols].to_numpy()
    comparison[f'{forecaster}_brier_soft'] = ((probabilities - soft_target) ** 2).sum(axis=1)
    comparison[f'{forecaster}_brier_majority'] = ((probabilities - majority_target) ** 2).sum(axis=1)
    summary_rows.append({
        'forecaster': forecaster,
        'items': len(comparison),
        'mean_brier_vs_rater_proportions': comparison[f'{forecaster}_brier_soft'].mean(),
        'mean_brier_vs_majority_one_hot': comparison[f'{forecaster}_brier_majority'].mean(),
        'majority_matches': (comparison[f'{forecaster}_label'] == comparison['majority_label']).sum(),
    })
forecaster_summary = pd.DataFrame(summary_rows)
display(comparison)
forecaster_summary

### Your response — Problem 3(c)

Type your response here **or** insert a clear image of handwritten work here.

**Prediction before assistance/output, when requested:**  
TODO or not applicable

**Analysis, evidence, or reasoning:**  
TODO

**Typed statistical conclusion (required even with handwriting):**  
TODO

d. **Measurement audit and synthesis (4 points).** Select one case where the
models disagree and quote only the shortest phrase needed to identify its
ambiguity. For that case, report $\widehat q$, $Y_{\mathrm{maj}}$, and the three
forecasters' highest-probability labels. Explain what the soft target records
that the majority target discards. Use one specific fact from each linked model
   card as a plausible—not proven—explanation for the differing forecasts.

   Finally, organize the data as one row per sampled sentence $X_j$, with the
   student and two model forecasts in three aligned columns. Explain (i) under
   what sampling conditions the sentence-level rows could be iid, (ii) why the
   three forecasts within a row are not justified as independent, identically
   distributed rater draws, and (iii) why the student and two models are not
   exchangeable. Your explanation should use the fact that all three columns
   respond to the same random sentence, while fixed model outputs are
   deterministic conditional on $X_j=x_j$.

## Required AI-use record (2 points)

Complete the AI-use table in the starter notebook for Problems 2(e) and 3. For Problem 2(e), record
the **initial prompt only** rather than the whole conversation. For Problem 3,
record both checkpoint IDs and revisions; write “no natural-language prompt”
for the prompt field. For both uses, record the purpose, what you checked, what
changed, and one decision you retained. Preserve Problem 3's raw probabilities
separately.

| Assignment part | Tool or checkpoint | Purpose | Initial prompt only | What I checked | What changed after checking | Decision I remained responsible for |
|---|---|---|---|---|---|---|
| Problem 2(e) | TODO | TODO | TODO | TODO | TODO | TODO |
| Problem 3 | Cardiff revision `3216a57f`; DistilBERT revision `cf991100` | Generate two frozen probability forecasts | no natural-language prompt | TODO | TODO | TODO |

### Your response — Problem 3(d)

Type your response here **or** insert a clear image of handwritten work here.

**Prediction before assistance/output, when requested:**  
TODO or not applicable

**Analysis, evidence, or reasoning:**  
TODO

**Typed statistical conclusion (required even with handwriting):**  
TODO

## Final submission check

- [ ] I restarted the runtime and ran all cells from top to bottom.
- [ ] Every requested denominator, table, figure, excerpt, and interpretation is visible.
- [ ] Any handwritten images are legible and each has a typed description or statistical conclusion.
- [ ] Required raw AI prompts/outputs and the AI-use record are preserved.
- [ ] I opened my downloaded PDF and `.ipynb` before uploading them.

The PDF is the primary grading surface; the notebook is the executable record. Both represent the same work.